# Stability Analysis - Demo_BESS Parameter Sweep

This notebook sweeps one parameter of `Demo_BESS`, linearizes the model at each value, and analyzes the eigenvalues to track how the modes and their stability change across the sweep.

Set `REINITIALIZE_EACH_CASE` to choose whether the model is initialized once before the sweep, or reinitialized after each parameter update.

In [ ]:
import logging
import os
import shutil
import sys

sys.path.append(os.path.abspath(".."))
sys.path.append(os.path.abspath("scripts"))

logging.getLogger("OMPython").setLevel(logging.ERROR)

import numpy as np
import pandas as pd
from OMPython import OMCSessionLocal

from scripts.helpers import *
from scripts.parametric_study_helpers import case_label, load_modelica_file
from scripts.single_model_initialization import initialize_loaded_model
from linearize_scripting import linearize_scripting

## User Configuration

Choose the model, the parameter to sweep and its values, and the linearization settings here.

In [ ]:
BASE_MODEL = "Demo_BESS"
BASE_MODEL_FILE = os.path.abspath(os.path.join("models", "Demo_BESS.mo"))

OMLIB_DIR = os.path.join(os.path.expanduser("~"), ".openmodelica", "libraries")
os.environ["OPENMODELICALIBRARY"] = OMLIB_DIR
MODELICA_PKG_PATH = os.path.join(OMLIB_DIR, "Modelica 3.2.3+maint.om", "package.mo")
DYNAWO_PKG_PATH = os.path.abspath(
    os.path.join("..", "dynawo_library", "Dynawo", "package.mo")
)

SWEEP_COMPONENT = "BESS"
SWEEP_PARAMETER = "Kpg"
SWEEP_VALUES = [1.0, 2.0, 3.0, 5.0, 7.0, 10.0]

REINITIALIZE_EACH_CASE = True

LINEARIZATION_TIME = "2.0"
MODE_TOL = 1e-8

INIT_MODEL_BY_COMPONENT = {}

# Leave empty to disable slack-specific handling.
SLACK_COMPONENT = ""

OUTPUT_DIR = os.path.abspath(os.path.join("outputs", "Demo_BESS_Kpg_stability_sweep"))

## Prepare OpenModelica Session

The output folders are created and the library paths are checked before the run.

In [ ]:
BUILD_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "omc-build")

if not os.path.isfile(BASE_MODEL_FILE):
    raise RuntimeError(f"Base model file not found: {BASE_MODEL_FILE}")
if not os.path.isfile(DYNAWO_PKG_PATH):
    raise RuntimeError(f"Dynawo package not found: {DYNAWO_PKG_PATH}")
if not os.path.isfile(MODELICA_PKG_PATH):
    raise RuntimeError(f"Modelica package not found: {MODELICA_PKG_PATH}")

shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
os.makedirs(BUILD_OUTPUT_DIR, exist_ok=True)

StudyOMC = OMCSessionLocal()
load_modelica_file(
    StudyOMC,
    BASE_MODEL_FILE,
    MODELICA_PKG_PATH,
    DYNAWO_PKG_PATH,
)
check_user_configuration_single(
    StudyOMC,
    model=BASE_MODEL,
    sweep_component=SWEEP_COMPONENT,
    sweep_parameter=SWEEP_PARAMETER,
    slack_component=SLACK_COMPONENT,
    init_model_by_component=INIT_MODEL_BY_COMPONENT,
)

## Initialize the Base Case

When `REINITIALIZE_EACH_CASE` is `false`, initialize the base model once here and reuse it for every sweep case. When it is `true`, this cell only closes the session opened above.

In [ ]:
if not REINITIALIZE_EACH_CASE:
    base_initialized_case = initialize_loaded_model(
        StudyOMC,
        source_model=BASE_MODEL,
        case_name=BASE_MODEL,
        output_dir=OUTPUT_DIR,
        modelica_package_path=MODELICA_PKG_PATH,
        dynawo_package_path=DYNAWO_PKG_PATH,
        init_model_by_component=INIT_MODEL_BY_COMPONENT,
        slack_component=SLACK_COMPONENT,
    )

    base_initialized_model = base_initialized_case["initialized_model"]
    base_initialized_model_file = base_initialized_case["initialized_model_file"]
    print("Base case initialized once:", base_initialized_model)

StudyOMC.sendExpression("quit()")

## Run the Parameter Sweep

For each parameter value the model is initialized, linearized, and its state-space matrix is stored.

In [ ]:
A_matrices = {}
linear_states_by_case = {}
linearization_sizes = []

for value in SWEEP_VALUES:
    label = case_label(value)
    parameter_name = SWEEP_PARAMETER.replace(".", "_")
    case_model = "_".join([BASE_MODEL, SWEEP_COMPONENT, parameter_name, label])
    case_build_dir = os.path.join(BUILD_OUTPUT_DIR, case_model)
    os.makedirs(case_build_dir, exist_ok=True)

    StudyOMC = OMCSessionLocal()

    try:
        if REINITIALIZE_EACH_CASE:
            load_modelica_file(
                StudyOMC,
                BASE_MODEL_FILE,
                MODELICA_PKG_PATH,
                DYNAWO_PKG_PATH,
            )

            omc_call(
                StudyOMC,
                f"setParameterValue({BASE_MODEL}, {SWEEP_COMPONENT}.{SWEEP_PARAMETER}, {value})",
                parsed=False,
            )

            initialized_case = initialize_loaded_model(
                StudyOMC,
                source_model=BASE_MODEL,
                case_name=case_model,
                output_dir=OUTPUT_DIR,
                modelica_package_path=MODELICA_PKG_PATH,
                dynawo_package_path=DYNAWO_PKG_PATH,
                init_model_by_component=INIT_MODEL_BY_COMPONENT,
                slack_component=SLACK_COMPONENT,
            )

            linearized_model = initialized_case["initialized_model"]
        else:
            load_modelica_file(
                StudyOMC,
                base_initialized_model_file,
                MODELICA_PKG_PATH,
                DYNAWO_PKG_PATH,
            )

            omc_call(
                StudyOMC,
                f"setParameterValue({base_initialized_model}, {SWEEP_COMPONENT}.{SWEEP_PARAMETER}, {value})",
                parsed=False,
            )

            linearized_model = base_initialized_model

        matrices = linearize_scripting(
            StudyOMC,
            linearized_model,
            startTime=0.0,
            stopTime=float(LINEARIZATION_TIME),
            stepSize=0.001,
            tolerance=1e-6,
            outdir=case_build_dir,
        )

        A_matrices[value] = matrices["A"]
        linear_states_by_case[value] = matrices["states"]

        if value == SWEEP_VALUES[0]:
            linearization_sizes.append({
                "A": "x".join(str(d) for d in matrices["A"].shape),
                "B": "x".join(str(d) for d in matrices["B"].shape),
                "C": "x".join(str(d) for d in matrices["C"].shape),
                "D": "x".join(str(d) for d in matrices["D"].shape),
            })
    finally:
        StudyOMC.sendExpression("quit()")

    print(f"Finished {SWEEP_COMPONENT}.{SWEEP_PARAMETER} = {value}")

linear_states = linear_states_by_case[SWEEP_VALUES[0]]

for value in SWEEP_VALUES:
    if linear_states_by_case[value] != linear_states:
        raise RuntimeError(f"Linear state order changed for {SWEEP_COMPONENT}.{SWEEP_PARAMETER} = {value}")

print("Linearization sweep finished.")

## Linearization Summary

The dimensions of the linearized system are reported for one case as a sanity check.

In [ ]:
sizes = linearization_sizes[0]

linearization_summary = pd.DataFrame([{
    "n_states": len(linear_states),
    "size_A": sizes["A"],
    "size_B": sizes["B"],
    "size_C": sizes["C"],
    "size_D": sizes["D"],
}])

display(linearization_summary)

## Analyze Modes

The modes of each case are computed and classified as stable, marginal, or unstable.

In [ ]:
mode_rows = []
summary_rows = []

for value in SWEEP_VALUES:
    A = A_matrices[value]
    # eigvals() uses NumPy's linear algebra module to calculate the eigenvalues of matrix A.
    eigenvalues = np.linalg.eigvals(A)
    eigenvalues = eigenvalues[np.lexsort((eigenvalues.imag, eigenvalues.real))]
    # Stability depends on the sign of the eigenvalue's real part.
    real_parts = eigenvalues.real

    for mode_index, eigenvalue in enumerate(eigenvalues, start=1):
        real_part = eigenvalue.real
        imag_part = eigenvalue.imag
        # We use a small tolerance margin (MODE_TOL) to discard numerical noise from the solver.
        if real_part > MODE_TOL:
            stability = "unstable"
        elif real_part < -MODE_TOL:
            stability = "stable"
        else:
            stability = "marginal"

        mode_rows.append({
            "parameter_value": value,
            "mode": mode_index,
            "real_part": real_part,
            "imag_part": imag_part,
            "stability": stability,
        })

    summary_rows.append({
        "parameter_value": value,
        "max_real_part": real_parts.max(),
        "unstable_modes": int((real_parts > MODE_TOL).sum()),
        "marginal_modes": int((np.abs(real_parts) <= MODE_TOL).sum()),
        "stable_modes": int((real_parts < -MODE_TOL).sum()),
    })

mode_df = pd.DataFrame(mode_rows).sort_values(["parameter_value", "mode"], ignore_index=True)
stability_summary = pd.DataFrame(summary_rows).sort_values("parameter_value", ignore_index=True)

print("Mode analysis finished.")

## Check Unstable Modes

The modes that become unstable across the sweep are identified.

In [ ]:
unstable_modes = []

for mode in sorted(mode_df["mode"].unique()):
    rows_for_mode = mode_df[mode_df["mode"] == mode]

    if (rows_for_mode["stability"] == "unstable").any():
        unstable_modes.append(mode)

unstable_mode_table = pd.DataFrame({"mode": unstable_modes})

for value in SWEEP_VALUES:
    column_name = f"{SWEEP_PARAMETER}={value}"
    stability_values = []

    for mode in unstable_modes:
        rows_for_mode = mode_df[(mode_df["mode"] == mode) & (mode_df["parameter_value"] == value)]
        stability_values.append(rows_for_mode["stability"].iloc[0])

    unstable_mode_table[column_name] = stability_values

unstable_mode_rows = mode_df["mode"].isin(unstable_modes)
unstable_mode_df = mode_df[unstable_mode_rows].sort_values(["mode", "parameter_value"], ignore_index=True)

print("Unstable-mode check finished.")

## Modes

The stability summary and the unstable modes are displayed.

In [ ]:
print("Stability summary")
display(stability_summary)

print("Modes that become unstable")
display(unstable_mode_table)

## Mode Plots

The unstable modes are plotted in the complex plane across the sweep.

In [ ]:
import matplotlib.pyplot as plt


if not unstable_modes:
    print("No modes become unstable for these sweep values.")
else:
    # symmetric limits centred on the origin so all four quadrants are visible
    r_lim = 1.1 * unstable_mode_df["real_part"].abs().max()
    i_lim = 1.1 * unstable_mode_df["imag_part"].abs().max()

    fig, ax = plt.subplots()
    ax.set_xlabel("real(lambda)")
    ax.set_ylabel("imag(lambda)")
    ax.set_title(f"Eigenvalue evolution vs {SWEEP_COMPONENT}.{SWEEP_PARAMETER}")
    ax.set_xlim(-r_lim, r_lim)
    ax.set_ylim(-i_lim, i_lim)
    ax.axvline(0.0, linestyle="--", color="black")  # Re = 0  (stability boundary)
    ax.axhline(0.0, linestyle="--", color="black")  # Im = 0

    for mode in unstable_modes:
        mode_data = unstable_mode_df[unstable_mode_df["mode"] == mode]
        ax.plot(mode_data["real_part"], mode_data["imag_part"], marker="o", label=f"mode {mode}")
        for _, row in mode_data.iterrows():
            ax.annotate(
                row["parameter_value"],
                (row["real_part"], row["imag_part"]),
                fontsize=8,
                verticalalignment="bottom",
            )

    ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0))